# Damaten Hex AI と対戦（ブラウザ上）

公開リポジトリ **Koushien552/Damaten** の強いエンジン（PUCT + MCTS-Solver、学習済み **CNN** モデル対応）と、Colab 上で対局します。

**使い方**: 上のセルから順に実行 → 最後のセルに出る盤面で対局。**GPU不要・CPUで動作**します。

- **B（Black / ●）**: 上辺 ⇕ 下辺 をつなぐ（先手）
- **W（White / ◯）**: 左辺 ⇔ 右辺 をつなぐ（後手）
- 着手入力例: `e5`（左=列の英字、右=行の数字）→「打つ」かEnter
- 「AI強さ」スライダー＝探索量（大きいほど強く・遅い）。「新規対局」で手番/設定を反映してリセット。

> メモ: AIの思考は私たちのエンジン（`hexai move --model ...`）に丸投げしています。Banana版の弱いC++ではなく、CNNを読み込んだ本番エンジンが相手です。

In [ ]:
#@title 1) 公開リポジトリとモデルを取得（GitHubトークン不要）
import os, subprocess

REPO = 'Damaten'
# Clone without pulling LFS content (skip the ~0.8GB of self-play TSV);
# we only need the model object.
env = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Koushien552/Damaten.git'], check=True, env=env)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], env=env)

subprocess.run(['git', '-C', REPO, 'lfs', 'install'], check=True)
# Fetch ONLY the trained model (not the whole self-play dataset).
subprocess.run(['git', '-C', REPO, 'lfs', 'pull', '--include', 'models/hex_model.nn'], check=True)

hdr = open(os.path.join(REPO, 'models', 'hex_model.nn'), 'r', errors='replace').readline().strip()
print('model header:', hdr)
assert hdr.startswith('HEX'), 'モデルのダウンロードに失敗しました（LFS）。'
print('OK: モデル準備完了（HEXCNN_V1=CNN / HEXNN_V1=MLP）')

In [ ]:
#@title 2) エンジンをコンパイル（AVX2でCNN推論を高速化）
import os, subprocess

def build(extra):
    return subprocess.run(['g++', '-O3', '-std=c++17'] + extra +
                          ['-o', 'Damaten/hexai', 'Damaten/src/main.cpp'],
                          capture_output=True, text=True)

r = build(['-mavx2', '-mfma'])
if r.returncode != 0:
    print('AVX2 build failed, retrying without AVX2...')
    print(r.stderr[-1500:])
    r = build([])

if r.returncode == 0 and os.path.exists('Damaten/hexai'):
    print('コンパイル成功: Damaten/hexai')
else:
    print('コンパイル失敗')
    print(r.stderr[-2000:])

In [ ]:
import subprocess, re
from collections import deque
import ipywidgets as widgets
from IPython.display import display

N = 9
EMPTY, BLACK, WHITE = 0, 1, 2
ENGINE = 'Damaten/hexai'
MODEL  = 'Damaten/models/hex_model.nn'

class GameState:
    def __init__(self, board=None, cur=BLACK, last_move=None):
        self.board = board if board is not None else [EMPTY] * (N * N)
        self.cur = cur
        self.last_move = last_move

    def apply(self, m):
        nb = self.board[:]; nb[m] = self.cur
        return GameState(nb, WHITE if self.cur == BLACK else BLACK, m)

    def has_won(self, player):
        vis = [False] * (N * N); q = deque()
        dr = (-1, -1, 0, 0, 1, 1); dc = (0, 1, -1, 1, -1, 0)
        if player == BLACK:
            for c in range(N):
                if self.board[c] == BLACK: vis[c] = True; q.append(c)
        else:
            for r in range(N):
                i = r * N
                if self.board[i] == WHITE: vis[i] = True; q.append(i)
        while q:
            v = q.popleft(); rv, cv = divmod(v, N)
            if player == BLACK and rv == N - 1: return True
            if player == WHITE and cv == N - 1: return True
            for d in range(6):
                nr, nc = rv + dr[d], cv + dc[d]
                if 0 <= nr < N and 0 <= nc < N:
                    ni = nr * N + nc
                    if self.board[ni] == player and not vis[ni]:
                        vis[ni] = True; q.append(ni)
        return False

    def is_terminal(self):
        return self.has_won(BLACK) or self.has_won(WHITE)

    def legal_moves(self):
        return [i for i, x in enumerate(self.board) if x == EMPTY]

# AI thinking is delegated to our strong engine (PUCT + MCTS-Solver + CNN model)
def engine_move(gs, iters):
    board_str = ''.join(str(x) for x in gs.board)
    last = gs.last_move if gs.last_move is not None else -1
    args = [ENGINE, 'move', '--n', str(N), '--board', board_str,
            '--player', str(gs.cur), '--last', str(last),
            '--model', MODEL, '--iters', str(iters)]
    r = subprocess.run(args, capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        raise RuntimeError('engine error: ' + (r.stderr.strip() or ('exit %d' % r.returncode)))
    return int(r.stdout.strip().split()[0])

def idx_to_str(m):
    return chr(ord('a') + m % N) + str(m // N + 1)

def str_to_idx(s):
    s = s.strip().lower()
    a = re.search(r'[a-z]', s); b = re.search(r'[0-9]+', s)
    if not a or not b: raise ValueError
    c = ord(a.group(0)) - ord('a'); r = int(b.group(0)) - 1
    if not (0 <= r < N and 0 <= c < N): raise ValueError
    return r * N + c

# ---------- state ----------
gs = GameState()
human_player = BLACK
ai_player = WHITE

# ---------- widgets ----------
status = widgets.HTML()
board_view = widgets.HTML()
move_input = widgets.Text(placeholder='例: e5', description='手:', layout=widgets.Layout(width='200px'))
play_btn = widgets.Button(description='打つ', button_style='success', layout=widgets.Layout(width='80px'))
color_dd = widgets.Dropdown(options=[('あなた=先手 ●', BLACK), ('あなた=後手 ◯', WHITE)], value=BLACK, layout=widgets.Layout(width='190px'))
iters_sl = widgets.IntSlider(value=1500, min=200, max=4000, step=100, description='AI強さ', continuous_update=False, layout=widgets.Layout(width='330px'))
new_btn = widgets.Button(description='新規対局', button_style='primary', layout=widgets.Layout(width='100px'))

def render():
    rows = []
    header = '&nbsp;&nbsp;&nbsp;&nbsp;' + '  '.join(chr(ord('a') + i) for i in range(N))
    rows.append("<pre style='font-size:18px;line-height:1.35;margin:0;'>" + header + "</pre>")
    for r in range(N):
        indent = '&nbsp;' * r; cells = []
        for c in range(N):
            x = gs.board[r * N + c]
            if x == BLACK: cells.append("<span style='color:#1a6fcc;font-weight:700;'>●</span>")
            elif x == WHITE: cells.append("<span style='color:#cc2a1a;font-weight:700;'>◯</span>")
            else: cells.append("<span style='color:#bbb;'>＋</span>")
        line = indent + ('%2d ' % (r + 1)) + ' '.join(cells)
        rows.append("<pre style='font-size:18px;line-height:1.35;margin:0;'>" + line + "</pre>")
    board_view.value = ''.join(rows)

def upd(msg=''):
    if gs.is_terminal():
        win = 'Black ●' if gs.has_won(BLACK) else 'White ◯'
        who = 'あなたの勝ち！' if gs.has_won(human_player) else 'AIの勝ち'
        status.value = '<b>終局:</b> ' + win + ' — ' + who + '　' + msg
    else:
        turn = 'あなた' if gs.cur == human_player else 'AI'
        status.value = '<b>手番:</b> ' + turn + '　' + msg

def refresh():
    render(); upd()

def ai_move():
    global gs
    if gs.is_terminal() or gs.cur != ai_player: return
    status.value = '<b>AI 思考中...</b>'
    try:
        m = engine_move(gs, iters_sl.value)
    except Exception as e:
        status.value = "<span style='color:red'>AIエラー: " + str(e) + "</span>"; return
    if m is not None and m >= 0:
        gs = gs.apply(m); upd('AI → ' + idx_to_str(m))

def do_human_move(_=None):
    global gs
    if gs.is_terminal() or gs.cur != human_player: return
    raw = move_input.value; move_input.value = ''
    try:
        m = str_to_idx(raw)
    except Exception:
        upd('無効な入力です。例: e5'); return
    if gs.board[m] != EMPTY:
        upd('そのマスは埋まっています'); return
    gs = gs.apply(m); refresh()
    if not gs.is_terminal() and gs.cur == ai_player:
        ai_move()
    refresh()

def new_game(_=None):
    global gs, human_player, ai_player
    human_player = color_dd.value
    ai_player = WHITE if human_player == BLACK else BLACK
    gs = GameState(); refresh()
    if gs.cur == ai_player:
        ai_move(); refresh()

play_btn.on_click(do_human_move)
new_btn.on_click(new_game)
try:
    move_input.on_submit(do_human_move)   # Enter key (older ipywidgets only)
except Exception:
    pass

display(widgets.HBox([color_dd, new_btn]), iters_sl, status, board_view, widgets.HBox([move_input, play_btn]))
refresh()